# Computing the random teams and their stats

We assume that the steps of [GettingDex](GettingDex.md) have been followed, and that the saved Pokedex is written in `/data/pokedex_raw.json`. 

In [ ]:
import json

# 'keying' POKEDEX_raw entries by `id`
with open('./../data/pokedex_raw.json','r') as file:
    POKEDEX_raw = json.load(file)

POKEDEX = { item['id'] : {key:item.get(key) for key in item.keys()} for item in POKEDEX_raw }

# saving to file
with open('./../data/pokedex_for_test.json', 'w') as file:
    json.dump(POKEDEX, file)

### Actual computation
<span style="color:orange">Note:</span> Don't forget that the following assumes that (following [ComputingTeams](ComputingTeams.md)) the "team generator sever" is running/listening on port `localhost:3000`.

In [ ]:
import json
from pathlib import Path
from team_compute import get_player_dets, get_teams_full, append_team_stats

replay_dir = Path("../data/test_data_replays/")

with open('../data/POKEDEX_for_test.json','r') as file:
    DEX = json.load(file)

In [ ]:
errs = []

for replay in replay_dir.glob("*.json"): 
    try:
        with replay.open() as file:
            battle_json = json.load(file)

        battle_json['player_dets'] = get_player_dets(battle_json)
        
        TEAMS = get_teams_full(battle_json)
        try :
            for team in TEAMS:
                append_team_stats(DEX, team, battle_json['id']) # `id` is included just for debugging purposes
        except : 
            errs.append(replay.name)
            continue
        battle_json['teams_full'] = TEAMS
        
        replay.write_text(json.dumps(battle_json), encoding='utf-8')
    
    except:
        print("Error parsing file: %s" % replay.name)
        errs.append(replay.name)
        continue

errs

# Parsing battles into `pandas.DataFrame` and removing custom-rule battles

Naturally, the following can be modified and run in different directories.

In [ ]:
from battle import *
from bat_to_list import battle_to_list

replay_dir = Path("../data/test_data_replays/")

# ===========================
DATA = []
customs = []
errs = []

for file in replay_dir.glob("*.json") : 
    try : 
        bat = Battle(file, parse=True)
        
        if not bat.custom_ruleQ : 
            DATA.append(battle_to_list(bat))
        else : 
            customs.append(file.name)
    except : 
        print(f"error with {file.name}")
        errs.append(file.name)
        continue

print(customs)
print(errs)

0

In [ ]:
# delete any replays having custom rules
import os
for replay in customs : 
    os.remove(replay_dir / file.name)

In [ ]:
import pandas as pd 

with open('../data/data_col_names.txt','r') as file:
    col_names = eval(file.read())

df = pd.DataFrame(DATA, columns=col_names)
df.info()

In [ ]:
with open('./../data/test_data_cleaned.csv','w') as file:
    file.write(df.to_csv(index=False))
df = pd.read_csv("./../data/test_data_cleaned.csv")
df.info() # testing